# From LLM to Agent: reading the scientific literature

**AgentForge Workshop · Tutorial 1 — Agent frameworks**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentForgeWorkshop/Tutorials/blob/main/Tutorial1_frameworks/01_agents_for_science.ipynb)

By the end of this notebook you will have built an assistant that searches arXiv for real
papers on a topic you choose, summarises them, writes a short review to a file, and returns a
literature table you can drop into a spreadsheet.

### The one idea

Every agent **framework** — LangChain, DeepAgents, Pydantic AI, CrewAI, all of them — wraps
the same short loop:

> Send the conversation **plus a list of tool descriptions** to the LLM.
> If it replies asking to call a tool, **run that Python function**, append the result to the
> conversation, and call the LLM again. Stop when it replies with plain text.

That loop *is* the agent. You will write it yourself in Step 2, in about twenty lines, before
you touch a framework. After that, every framework is "the loop, plus conveniences".


### Outline

| Step | What |
|---|---|
| 0 | Setup, and watching a bare LLM fail at a literature question |
| 1 | A tool is just a Python function (`search_arxiv`) |
| 2 | **The loop**, written by hand |
| 3 | The same agent in LangChain, in four lines |
| 4 | A second tool, so the agent has to plan |
| 5 | Structured output: a literature table, not prose |
| 6 | Exercises |
| 7 | Finale: DeepAgents, and the caveats that matter |

### What you need

* A Google account (for Colab) and an **OpenRouter API key** — one minute to create:
  <https://openrouter.ai/keys>.


---
## Step 0 — Setup

Run the next cell once. It takes about a minute.

Note: The "pip dependency google-colab" error is expected, but the notebook works fine.

In [19]:
%pip install -q "langchain==1.4.2" "langchain-google-genai==4.4.0" \
               "langchain-openai==1.6.2" "deepagents==0.7.16"

### Your API key

1. Open <https://openrouter.ai/keys> and create a key. Add a little credit to the account —
   this notebook costs well under a cent to run end to end.
2. In Colab, click the **🔑 key icon** in the left sidebar (Secrets).
3. Add a secret named exactly `OPENROUTER_API_KEY`, paste your key as the value, and turn on
   **Notebook access**.

Why Secrets and not a variable in a cell? Because a key pasted into a cell gets saved into the
notebook, and notebooks get shared. Treat an API key like a password.

In [20]:
import getpass
import os

from langchain.chat_models import init_chat_model


def get_key(name: str) -> str:
    """Read an API key from Colab's Secrets panel, or ask for it if there is none."""
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:  # not in Colab, or that secret is not set
        return os.environ.get(name) or getpass.getpass(f"{name}: ")


PROVIDER = "openrouter"  # <-- "openrouter" or "google"

if PROVIDER == "openrouter":
    # Any id from https://openrouter.ai/models, prefixed with "openai:" because OpenRouter
    # speaks the OpenAI API. It MUST support tool calling -- filter for "Tools" on that page.
    MODEL_NAME = "openai:deepseek/deepseek-v4.1-flash"
    # MODEL_NAME = "openai:nvidia/nemotron-3-ultra-550b-a55b:free"
    model = init_chat_model(
        MODEL_NAME,
        temperature=0,
        base_url="https://openrouter.ai/api/v1",
        api_key=get_key("OPENROUTER_API_KEY"),
    )

elif PROVIDER == "google":
    MODEL_NAME = "google_genai:gemini-3.5-flash"
    os.environ["GOOGLE_API_KEY"] = get_key("GOOGLE_API_KEY")
    model = init_chat_model(MODEL_NAME, temperature=0)

print("Model ready:", MODEL_NAME)

Model ready: openai:deepseek/deepseek-v4.1-flash


**Using a different provider?** Change `PROVIDER` and `MODEL_NAME` above.

### Choosing a model on OpenRouter

OpenRouter is a single API in front of hundreds of models, and some carry a `:free` suffix.
Two warnings, because this notebook is *nothing but* tool calls:

* **The model must support tool calling.** Filter for **Tools** (in the left sidebar **Supported parameters**) on
  <https://openrouter.ai/models>.
* Free models are heavily rate-limited and often queue.

In [21]:
# Smoke test: one call, to prove the key works.
#
# Google's SDK logs a one-off warning here about "automatic function calling (AFC)".
# Nothing is wrong: it appears once per session, and Step 3 explains what it means.
print(model.invoke("Reply with exactly: ready").text)

ready


---
## Step 0b — Why we need tools at all

Pick a topic you actually care about. Everything below uses it, so choose something from your
own field — the workshop is more interesting when the papers are ones you can judge.

In [22]:
TOPIC = "machine learning interatomic potentials for catalysis"  # <-- change me

In [23]:
answer = model.invoke(
    f"List 5 papers on '{TOPIC}' that were posted to arXiv in the last two weeks. "
    "Give the title, the authors and the arXiv link for each."
)
print(answer.text)

I can’t reliably list 5 papers posted to arXiv in the last two weeks (roughly **2026-04-25 to 2026-05-09**) because I don’t have live web/arXiv access in this environment, and my training data does not cover that period. I won’t fabricate titles, authors, or arXiv links.

You can retrieve exactly what you need with this arXiv API query/script. It searches for ML interatomic potentials + catalysis and filters to the last 14 days:

```python
import urllib.request, urllib.parse
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone

terms = (
    '(all:"machine learning interatomic potential" OR '
    'all:"machine learning interatomic potentials" OR '
    'all:"MLIP" OR all:"neural network potential") '
    'AND (all:catalysis OR all:catalytic OR all:catalyst)'
)

url = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode({
    "search_query": terms,
    "start": 0,
    "max_results": 100,
    "sortBy": "submittedDate",
    "sortOrder": "descending

### Look closely at what just happened

You will have got one of two things, and both are the point:

* **A refusal** — "I cannot browse the web, my knowledge has a cutoff date". Honest, useless.
* **Confident, plausible, unverifiable output** — titles that sound exactly like real papers,
  author lists of real researchers in the field, and arXiv IDs that may well resolve to a
  completely different paper. This is the failure mode that ends up in a bibliography and
  embarrasses you at review time.

The model is not lying. It has no connection to arXiv; it is completing text. Asking it for
this week's literature is like asking a colleague, locked in a room since their training
finished, to recite recent titles from memory.

**Fix: give it a way to look things up, and insist it only reports what it found.** That is all
a tool is.

---
## Step 1 — A tool is just a Python function

Here is our tool. Note what it is *not*: it is not a special framework object, it does not
inherit from anything, and there is no magic. It takes arguments, hits the public arXiv API
(no key needed), and returns a **string** — because a string is what we can paste into a
conversation with an LLM.

Read the signature and the docstring carefully, then skim the body; the XML parsing is the
boring part.

One detail in there is *not* boring, because it is the kind of thing that quietly ruins
results: **arXiv treats each search term as a single word.** Sending
`all:machine learning interatomic potentials` does not search for that phrase — arXiv reads
the first word, the rest becomes noise, and you get back whatever happened to be posted most
recently. The papers look plausible and have nothing to do with your topic.

`build_arxiv_query` translates a plain topic into arXiv's actual query language by ANDing the
meaningful words. Every data API has a quirk like this, and no amount of clever prompting
fixes it: **the tool has to be right before the agent can be.** When an agent gives you
confident nonsense, suspect the tool before you suspect the model.

In [24]:
import re
import textwrap
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

ARXIV_API = "https://export.arxiv.org/api/query"
ATOM = {"atom": "http://www.w3.org/2005/Atom"}

# Words that add nothing to a search and would only narrow the results for no reason.
STOPWORDS = {"a", "an", "and", "are", "as", "at", "based", "by", "for", "from", "in",
             "into", "is", "of", "on", "or", "the", "to", "use", "using", "via", "with"}


def build_arxiv_query(topic: str) -> str:
    """Translate a plain-English topic into arXiv's query language.

    arXiv treats each field term as a *single word*: `all:machine learning` is not a two-word
    search, and sending a whole topic that way quietly returns near-random recent papers.
    Terms have to be combined explicitly, so we AND the meaningful words together.
    """
    words = re.findall(r"[A-Za-z0-9][\w'-]*", topic)
    terms = [w for w in words if w.lower() not in STOPWORDS]
    return " AND ".join(f"all:{w}" for w in terms) or f"all:{topic}"


def search_arxiv(query: str, max_results: int = 5) -> str:
    """Search arXiv for recent papers and return their titles, authors, links and abstracts.

    Args:
        query: Search terms, e.g. 'graph neural network catalyst screening'.
        max_results: How many papers to return (1-10).
    """
    search_query = build_arxiv_query(query)
    params = urllib.parse.urlencode(
        {
            "search_query": search_query,
            "start": 0,
            "max_results": min(max_results, 10),
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        }
    )
    with urllib.request.urlopen(f"{ARXIV_API}?{params}", timeout=30) as response:
        feed = ET.fromstring(response.read())

    papers = []
    for entry in feed.findall("atom:entry", ATOM):
        title = " ".join(entry.findtext("atom:title", "", ATOM).split())
        abstract = " ".join(entry.findtext("atom:summary", "", ATOM).split())
        authors = [a.findtext("atom:name", "", ATOM) for a in entry.findall("atom:author", ATOM)]
        papers.append(
            f"Title: {title}\n"
            f"Authors: {', '.join(authors[:3])}{' et al.' if len(authors) > 3 else ''}\n"
            f"Published: {entry.findtext('atom:published', '', ATOM)[:10]}\n"
            f"Link: {entry.findtext('atom:id', '', ATOM)}\n"
            f"Abstract: {textwrap.shorten(abstract, 1000, placeholder=' ...')}"
        )

    if not papers:
        # Telling the agent what we actually sent lets it retry with fewer terms.
        return (
            f"No papers found for '{query}' (sent to arXiv as: {search_query}). "
            "Try fewer or broader search terms."
        )
    return "\n\n---\n\n".join(papers)

In [25]:
# No LLM involved here at all -- it is an ordinary function, so just call it.
print("Your topic:", TOPIC)
print("arXiv sees:", build_arxiv_query(TOPIC), "\n")

print(search_arxiv(TOPIC, max_results=2))

Your topic: machine learning interatomic potentials for catalysis
arXiv sees: all:machine AND all:learning AND all:interatomic AND all:potentials AND all:catalysis 



HTTPError: HTTP Error 429: Unknown Error

### The docstring is not documentation, it is part of the prompt

When we hand this function to a model, the framework turns it into a JSON schema — name,
description, argument names, argument descriptions — all taken from the **signature, type
hints and docstring**. That schema is what the model sees. It never sees the body.

So:

* `query: str` tells the model to send a string.
* `"Search terms, e.g. 'graph neural network catalyst screening'"` teaches it the *style* of
  query that works. Vague docstrings produce vague tool calls.
* `max_results: int = 5` gives it a default it can leave alone.

**Rewriting a docstring is a real way to change an agent's behaviour.** Come back and try it
once the agent is running.

In [26]:
# Curiosity check: this is exactly what the model gets shown.
import json

from langchain_core.utils.function_calling import convert_to_openai_tool

print(json.dumps(convert_to_openai_tool(search_arxiv), indent=2))

{
  "type": "function",
  "function": {
    "name": "search_arxiv",
    "description": "Search arXiv for recent papers and return their titles, authors, links and abstracts.",
    "parameters": {
      "properties": {
        "query": {
          "description": "Search terms, e.g. 'graph neural network catalyst screening'.",
          "type": "string"
        },
        "max_results": {
          "default": 5,
          "description": "How many papers to return (1-10).",
          "type": "integer"
        }
      },
      "required": [
        "query"
      ],
      "type": "object"
    }
  }
}


---
## Step 2 — The loop

This is the centre of the workshop. Everything after it is convenience.

```text
messages = [system prompt, user question]

repeat:
    reply = LLM(messages, tools=[search_arxiv])   # 1. ask
    if reply has no tool calls:                   # 2. plain text?
        return reply                              #    -> done
    for each tool call in reply:                  # 3. run the Python function
        messages.append(result)                   #    -> hand the result back
                                                  # 4. go round again
```

Two things worth saying out loud, because they surprise people:

* **The LLM never runs any code.** It emits a *request* — a name and some JSON arguments. Your
  Python runs the function. That is also your safety boundary: an agent can only do what its
  tools let it do.
* **The model has no memory between calls.** The `messages` list *is* the memory. Every turn we
  resend the whole conversation. That is also why long agent runs get expensive.

Below, `model.bind_tools([...])` just means "attach these tool schemas to every request". No
agent framework involved — we are using the chat model as a uniform way to talk to whichever
provider you picked, and writing the loop ourselves.

In [27]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

TOOLS = {"search_arxiv": search_arxiv}

SYSTEM_PROMPT = (
    "You are a research assistant for a scientist. "
    "Use the search_arxiv tool to find real papers before answering. "
    "Never invent titles, authors or links: report only what the tool returned, "
    "and always include the link for every paper you mention."
)


def run_agent(question: str, max_steps: int = 6) -> str:
    """The whole agent, in about twenty lines."""
    llm = model.bind_tools(list(TOOLS.values()))          # show the model what tools exist
    messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(question)]

    for step in range(1, max_steps + 1):
        reply = llm.invoke(messages)                      # 1. ask the model
        messages.append(reply)

        if not reply.tool_calls:                          # 2. plain text -> we are done
            print(f"[step {step}] answered in plain text\n")
            return reply.text

        for call in reply.tool_calls:                     # 3. run each requested tool
            print(f"[step {step}] calling {call['name']}({call['args']})")
            result = TOOLS[call["name"]](**call["args"])
            print(f"          -> {len(result)} characters of tool output")
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
                                                          # 4. round again, with the result
    return "Stopped: hit max_steps without a final answer."

In [28]:
print(run_agent(f"What are 3 recent papers on {TOPIC}? Summarise each in one sentence."))

[step 1] calling search_arxiv({'query': 'machine learning interatomic potentials catalysis', 'max_results': 10})


TimeoutError: The read operation timed out

### Read the trace, not just the answer

You should see something like: *step 1 calls `search_arxiv({...})`* → a few thousand characters
of tool output → *step 2 answered in plain text*. Two LLM calls, one function call. That is a
complete agent.

Notice what you controlled:

* **`max_steps`.** Without it, a confused model can loop forever, and each lap costs money.
  Every framework has this; it is usually called the recursion or iteration limit.
* **The system prompt.** "Never invent titles" is doing real work here. Try deleting that
  sentence later and see what changes.
* **What happens on failure.** Right now, if `search_arxiv` raises (network hiccup, odd
  characters in a query), the whole cell dies. A more forgiving version catches the exception
  and passes the error message back to the model as the tool result — which often lets it
  recover by trying a different query. That is one of the conveniences a framework gives you.

Now let us throw this away and get the same thing in four lines.

---
## Step 3 — The same agent, in LangChain

`create_agent(model, tools, system_prompt)` builds exactly the loop you just wrote.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model, tools=[search_arxiv], system_prompt=SYSTEM_PROMPT)

result = agent.invoke(
    {"messages": [HumanMessage(f"What are 3 recent papers on {TOPIC}? One sentence each.")]}
)
print(result["messages"][-1].text)

`result["messages"]` is the same conversation list you built by hand in Step 2 — the framework
kept it for you. Let us print it, so you can watch the agent's decisions rather than just read
its conclusion.

In [ ]:
def show_trace(result, width=400):
    """Print an agent run as a readable transcript."""
    for message in result["messages"]:
        kind = type(message).__name__.replace("Message", "").upper()
        for call in getattr(message, "tool_calls", None) or []:
            print(f"[{kind}] ---> {call['name']}({call['args']})")
        text = (message.text or "").strip()
        if text:
            print(f"[{kind}] {textwrap.shorten(text, width, placeholder=' ...')}")
        print()


show_trace(result)

### So what did the framework actually add?

Nothing you cannot write yourself — but a lot you would rather not write twice:

* the step limit, tool-error handling, and retries on flaky API calls;
* one interface across providers (that `MODEL_NAME` swap in Step 0);
* streaming, so a long run prints as it happens rather than after;
* conversation memory across turns (checkpointers), and trimming the history when it outgrows
  the context window;
* structured output (Step 5), and middleware to inspect or veto each step — which is how you
  add human approval before an agent does something irreversible.

> **Remember that `automatic function calling (AFC)` warning from Step 0?** It is this same
> question one layer further down. Google's SDK ships a loop of its own: hand it Python
> callables and it will run them for you and keep going. LangChain declines the offer — it
> sends only the tool *descriptions* and runs the loop itself, which is why you can see and
> control every step above. Two implementations of Step 2, politely disagreeing about who
> drives. The warning is logged once per session and changes nothing.

> ### ⚠️ A warning that will save you an hour
>
> Most LangChain tutorials you will find by searching are from 2023 and use the pre-1.0 API:
> `initialize_agent`, `AgentExecutor`, `ZeroShotAgent`, `LLMChain`, `Tool(...)` objects.
> That API is gone. If a snippet uses those names, it is stale — look for `create_agent` in the
> current docs instead: <https://docs.langchain.com/oss/python/langchain/agents>

---
## Step 4 — A second tool, and the agent starts planning

With one tool there is nothing to decide. Add a second and you can watch the model sequence its
work: *search first, then write the file with what I found.* Nobody wrote that order down.

In [ ]:
from pathlib import Path


def save_report(filename: str, content: str) -> str:
    """Save a markdown report so the scientist can read it later.

    Args:
        filename: File name ending in .md, e.g. 'catalysis_review.md'.
        content: The full markdown text of the report.
    """
    path = Path(filename)
    path.write_text(content, encoding="utf-8")
    return f"Saved {path.resolve()} ({len(content)} characters)."


reporter = create_agent(
    model,
    tools=[search_arxiv, save_report],
    system_prompt=(
        SYSTEM_PROMPT
        + " When asked for a report, search first, then save it with save_report. "
        "End every report with a '## References' section listing each link you cite."
    ),
)

result = reporter.invoke(
    {
        "messages": [
            HumanMessage(
                f"Find 4 recent papers on {TOPIC}, then save a short markdown report "
                "called 'report.md' with a one-paragraph overview and one bullet per paper."
            )
        ]
    }
)
show_trace(result)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(Path("report.md").read_text(encoding="utf-8")))

The interesting line in that trace is the second tool call: the agent passed the *entire report
text* as an argument, having composed it from the abstracts it had just read. It chose the
order, the content, and what went in the file.

It also, right now, has write access to this notebook's filesystem. That is worth sitting with
for a moment. **An agent can do exactly what its tools permit — no more, and no less.** When you
later give one a tool that submits jobs to a cluster, deletes files, or emails a collaborator,
that same sentence becomes your security model. Grant narrowly; log every call.

---
## Step 5 — Structured output: a table, not prose

Prose is a poor deliverable for science. You usually want rows: something to sort, filter, plot,
or hand to the next script. So describe the shape you want as Pydantic classes and pass it as
`response_format`. The framework asks the model to fill in that schema, and validates the
result before you ever see it.

In [ ]:
from typing import List

from pydantic import BaseModel, Field


class Paper(BaseModel):
    title: str = Field(description="Exact title, copied from the search results")
    link: str = Field(description="arXiv link, copied verbatim from the search results")
    method: str = Field(description="The method or system studied, in one short phrase")
    key_finding: str = Field(description="One sentence, in the authors' own terms")
    relevance: int = Field(description="1 = tangential, 5 = directly on topic", ge=1, le=5)


class LiteratureTable(BaseModel):
    """A small literature table built only from real search results."""

    topic: str
    papers: List[Paper]

In [ ]:
table_agent = create_agent(
    model,
    tools=[search_arxiv],
    system_prompt=SYSTEM_PROMPT,
    response_format=LiteratureTable,
)

result = table_agent.invoke(
    {"messages": [HumanMessage(f"Build a literature table of 5 recent papers on {TOPIC}.")]}
)

table = result["structured_response"]
print(type(table).__name__, "with", len(table.papers), "papers\n")
print(table.papers[0])

In [ ]:
import pandas as pd

df = pd.DataFrame([paper.model_dump() for paper in table.papers])
df.to_csv("literature_table.csv", index=False)  # now it is just data
df

`table` is a validated Python object, not text that looks like a table: `table.papers[0].link`
is a string you can follow, and a `relevance` of `"very high"` would have been rejected before
it reached you. Field descriptions steer the model exactly the way tool docstrings do.

One honest caveat: **`relevance` is the model's opinion, not a measurement.** It is a useful
sort key for triage and nothing more. Keep that distinction sharp in anything you publish.

If you switched to a small or free model and this step returns nonsense or fails outright, that
is the model rather than the code: filling a schema exactly is the most demanding thing this
notebook asks of one. It also makes a fair comparison — run this cell across three models and
you will see the gap that benchmarks are trying to describe.

---
## Step 6 — Your turn

Three exercises, roughly in order of difficulty. You have ten minutes; nobody finishes all
three.

### Exercise 1 — Make it yours (2 min)

Change `TOPIC` to something from your own work, re-run Step 5, and **check the links**. Open
two of them. Did the agent report them accurately? Did it quietly drop a paper?

### Exercise 2 — A second source (5 min)

Biology and medicine mostly do not live on arXiv. Write `search_pubmed(query, max_results)`
using the NCBI E-utilities (no key needed, but keep to 3 requests/second or fewer):

* search: `https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&retmode=json&term=...&retmax=...`
  → PubMed IDs, under `esearchresult.idlist`
* summarise: `https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pubmed&retmode=json&id=id1,id2`
  → titles, journals and dates, under `result.<id>`

Then pass `tools=[search_arxiv, search_pubmed]` and ask a question that spans both. Watch how
the agent decides which source to use — and remember that *your docstring* is what teaches it.

### Exercise 3 — A critic pass (10 min)

Agents get much more reliable when a second pass checks the first. Write `critique(draft,
evidence)`: send the draft summary **and the raw tool output** back to the model, and ask it to
list every claim in the draft that the evidence does not support. Run it on your Step 4 report.
This is the cheapest reliability win available to you.

In [ ]:
# Exercise 2 -- starter
import json
import urllib.parse
import urllib.request

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"


def search_pubmed(query: str, max_results: int = 5) -> str:
    """TODO: describe this tool for the model -- what it searches, and what a good query is.

    Args:
        query: TODO
        max_results: TODO
    """
    # 1. esearch  -> list of PubMed IDs
    # 2. esummary -> title / journal / date for those IDs
    # 3. return one readable string, the way search_arxiv does
    raise NotImplementedError


# pubmed_agent = create_agent(
#     model, tools=[search_arxiv, search_pubmed], system_prompt=SYSTEM_PROMPT
# )

In [ ]:
# Exercise 3 -- starter
def critique(draft: str, evidence: str) -> str:
    """Check a draft against the evidence it was supposedly built from."""
    prompt = (
        "Below is a DRAFT summary and the EVIDENCE it was built from.\n"
        "List every claim in the DRAFT that the EVIDENCE does not directly support. "
        "Quote the claim, then say what is missing. "
        "If everything checks out, reply 'No issues found'.\n\n"
        f"DRAFT:\n{draft}\n\nEVIDENCE:\n{evidence}"
    )
    return model.invoke(prompt).text


# evidence = search_arxiv(TOPIC, max_results=4)
# print(critique(Path("report.md").read_text(encoding="utf-8"), evidence))

---
## Step 7 — Finale: the same idea, scaled up

`create_deep_agent` from the [DeepAgents](https://docs.langchain.com/oss/python/deepagents/overview)
library has the same signature as `create_agent` and runs the same loop — but the agent it
builds comes with a to-do list it maintains as it works, a virtual filesystem it can write
drafts to, and the ability to spawn sub-agents that each research one sub-question and report
back. You change one function name.

In [ ]:
from deepagents import create_deep_agent

deep_agent = create_deep_agent(
    model=model,
    tools=[search_arxiv],
    system_prompt=(
        "You are a scientific literature analyst. Plan your work first, then search arXiv "
        "(several focused searches beat one broad one), and write your review to 'review.md' "
        "with the sections: Overview, Methods in use, Open questions, References. "
        "Cite only papers the search tool actually returned, with their links."
    ),
)

result = deep_agent.invoke(
    {"messages": [HumanMessage(f"Write a short review of recent work on {TOPIC}.")]}
)
show_trace(result, width=200)

In [ ]:
# DeepAgents keeps its files in the run's state rather than on disk.
for path, file in result.get("files", {}).items():
    print(f"========== {path} ==========")
    print(file["content"][:2500])

That run was slower and used considerably more tokens than Step 4, and the trace shows why:
planning steps, several searches, drafts rewritten. Which tells you when to reach for it.

| Use | Reach for |
|---|---|
| One clear task, a few tools, an answer you want now | `create_agent` |
| Open-ended work: survey a field, compare ten methods, multi-stage analysis | `create_deep_agent` |

DeepAgents is 0.x and moving quickly — pin the version in anything you rely on.

---
## Step 8 — Before you use this on real work

Six things, in roughly the order they will bite you.

1. **Verify every citation.** The agent read *abstracts*, not papers, and a fluent summary of an
   abstract can still misstate what the paper did. Click every link before it enters a
   manuscript. An agent-assisted bibliography is a draft, not a result.
2. **It is not reproducible by default.** Same prompt, same model, different day, different
   answer — `temperature=0` reduces this but does not remove it, and providers update models
   underneath you. If a result matters, save the prompt, the model version, the raw tool output
   and the run date alongside the output.
3. **Recency is not relevance.** We sorted by submission date, which means preprints: no peer
   review, no citation record. Sorting by relevance is one parameter change; deciding what
   belongs in a review is still your job.
4. **Cost and rate limits scale with steps, not questions.** Every lap of the loop resends the
   whole conversation. A ten-step run over long abstracts is not ten times one call — it is
   worse. Watch `max_steps`.
5. **Tools are permissions.** `save_report` can overwrite files. Anything you add later —
   deleting, submitting jobs, sending email, hitting an internal database — grants that power
   to a probabilistic system. Give the narrowest tool that does the job, and log its calls.
6. **Mind what you send.** Prompts and tool outputs leave your machine and go to a model
   provider. Unpublished results, patient data, anything under embargo or an IRB restriction:
   check your institution's policy first.

### Where to go next

* **Other frameworks, same loop.** [Pydantic AI](https://ai.pydantic.dev) (below) has the
  cleanest API of the bunch and the strongest typed-output story;
  [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/),
  [Google ADK](https://google.github.io/adk-docs/) and
  [smolagents](https://huggingface.co/docs/smolagents) are each worth an afternoon.
  [CrewAI](https://docs.crewai.com) and
  [Microsoft Agent Framework](https://learn.microsoft.com/agent-framework/) lead with
  multi-agent teams. The labels vary — framework, SDK, kit, harness — and the projects do
  not agree with each other; underneath, each one runs the loop from Step 2.
* **Full text, not abstracts.** [LlamaIndex](https://docs.llamaindex.ai) parses and indexes
  PDFs, so your agent can quote Section 3 instead of the abstract.
* **Seeing what your agent did.** [LangSmith](https://docs.langchain.com/langsmith/home) traces
  every step, token and cost — the first thing to add once runs stop being three steps long.
* **Tools you did not write.** The Model Context Protocol (MCP) lets an agent use tool servers
  someone else maintains, instead of your hand-rolled function.

### Optional — the same agent in Pydantic AI

Proof that the framework is the least important choice you make here: same tool, same model,
same loop, different spelling. Pydantic AI is async-first, so the call is `await agent.run(...)`
— which works at the top level of a Colab cell.

In [ ]:
%pip install -q "pydantic-ai==2.46.0"

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openrouter import OpenRouterProvider

pai_model = OpenAIChatModel(
    "deepseek/deepseek-v4.1-flash",
    provider=OpenRouterProvider(api_key=get_key("OPENROUTER_API_KEY")),
)

pai_agent = Agent(
    pai_model,  # with a Google key, the string "google:gemini-3.5-flash" works here instead
    tools=[search_arxiv],
    instructions=SYSTEM_PROMPT,
    output_type=LiteratureTable,  # the same Pydantic class from Step 5
)

pai_result = await pai_agent.run(f"Build a literature table of 3 recent papers on {TOPIC}.")
pd.DataFrame([paper.model_dump() for paper in pai_result.output.papers])

---

*AgentForge Workshop — Tutorial 1. Questions, fixes and better examples are welcome:
<https://github.com/AgentForgeWorkshop/Tutorials>*